# COMP 4330SEF — Whisper-small Fine-tuning on MCE Dataset

**目标**: 在 Mixed Cantonese-English (MCE) 数据集上，用 LoRA fine-tune Whisper-small，并与 Baseline 比较。

**主要结果**（已完成）:
- Baseline CER: 47.41%
- LoRA Fine-tuned CER: 17.30%（相对下降 63.5%）





## 1. 环境设定

挂载 Google Drive、安装套件、下载官方 MCE 数据集。


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install -q transformers datasets accelerate peft evaluate jiwer librosa soundfile gdown bitsandbytes


In [3]:
!gdown 1CFgHxTzYBKnIkRVBdCwlJXahZq3Zi87B -O MCE_dataset.zip
!unzip -q MCE_dataset.zip -d /content/MCE
print("MCE 数据集下载与解压完成")


Downloading...
From (original): https://drive.google.com/uc?id=1CFgHxTzYBKnIkRVBdCwlJXahZq3Zi87B
From (redirected): https://drive.google.com/uc?id=1CFgHxTzYBKnIkRVBdCwlJXahZq3Zi87B&confirm=t&uuid=47903992-8cc9-4362-a504-fb405dc43d97
To: /content/MCE_dataset.zip
100% 3.02G/3.02G [01:02<00:00, 48.3MB/s]
MCE 数据集下载与解压完成


## 2. Audio–Text 配对（自然排序）

使用自然排序（`1_2.wav` 在 `1_10.wav` 前面），确保音档与文字正确对齐。
跳过官方缺失的 speaker 85。


In [4]:
import os
import pandas as pd
import json
from pathlib import Path
from tqdm import tqdm
import librosa
import re

# ====================== 正确路径 ======================
AUDIO_ROOT = "/content/MCE/MCE_Dataset/Audio"
TEXT_ROOT  = "/content/MCE/MCE_Dataset/Text"
OUTPUT_JSON = "/content/mce_paired.json"
# =====================================================

def natural_sort_key(s):
    """自然排序: 让 1_2.wav 排在 1_10.wav 前面"""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', s)]

def get_audio_files(folder_path):
    audio_extensions = {'.wav', '.mp3', '.flac', '.m4a', '.ogg'}
    audio_files = []
    for f in os.listdir(folder_path):
        if Path(f).suffix.lower() in audio_extensions:
            audio_files.append(f)
    # 使用自然排序
    return sorted(audio_files, key=natural_sort_key)

def load_text_csv(csv_path):
    for encoding in ['utf-8', 'utf-8-sig', 'gbk', 'gb2312', 'big5', 'latin1']:
        try:
            df = pd.read_csv(csv_path, encoding=encoding)
            return df
        except:
            continue
    raise ValueError(f"无法读取 CSV: {csv_path}")

paired_data = []
missing_count = 0
success_folders = 0

print("开始配对（使用自然排序）...")

for i in tqdm(range(1, 161)):
    if i == 85:
        continue

    audio_folder = os.path.join(AUDIO_ROOT, f"{i}_MCE")
    text_csv = os.path.join(TEXT_ROOT, f"data_{i}.csv")

    if not os.path.exists(audio_folder) or not os.path.exists(text_csv):
        continue

    try:
        df = load_text_csv(text_csv)
    except Exception as e:
        print(f"读取 {text_csv} 失败: {e}")
        continue

    audio_files = get_audio_files(audio_folder)
    if len(audio_files) == 0:
        continue

    num_samples = min(len(audio_files), len(df))

    for idx in range(num_samples):
        audio_path = os.path.join(audio_folder, audio_files[idx])

        text = str(df.iloc[idx]['Instance']).strip()
        topic = str(df.iloc[idx]['Topic']).strip() if 'Topic' in df.columns else "unknown"

        try:
            duration = librosa.get_duration(path=audio_path)
            if duration < 0.3:
                continue
        except:
            missing_count += 1
            continue

        paired_data.append({
            "id": f"{i}_{idx}",
            "audio_path": audio_path,
            "text": text,
            "topic": topic,
            "duration": round(duration, 2)
        })

    success_folders += 1

print(f"\n========== 配对结果 ==========")
print(f"成功处理的资料夹数量：{success_folders}")
print(f"总共成功配对：{len(paired_data)} 笔资料")
print(f"无法读取的音档数量：{missing_count}")

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(paired_data, f, ensure_ascii=False, indent=2)

print(f"已储存到：{OUTPUT_JSON}")

print("\n=== 前 3 笔预览 ===")
for item in paired_data[:3]:
    print(item)


开始配对（使用自然排序）...


100%|██████████| 160/160 [00:10<00:00, 15.70it/s]



========== 配对结果 ==========
成功处理的资料夹数量：159
总共成功配对：14048 笔资料
无法读取的音档数量：0
已储存到：/content/mce_paired.json

=== 前 3 笔预览 ===
{'id': '1_0', 'audio_path': '/content/MCE/MCE_Dataset/Audio/1_MCE/1_1.wav', 'text': '"你知唔知今日嘅temperature會落到低至10度，一定要著厚d嘅clothes啊！"', 'topic': '天氣', 'duration': 8.11}
{'id': '1_1', 'audio_path': '/content/MCE/MCE_Dataset/Audio/1_MCE/1_2.wav', 'text': '"我check咗天氣app，佢話今日會有thunderstorm，最好唔好去戶外行嘅。"', 'topic': '天氣', 'duration': 8.7}
{'id': '1_2', 'audio_path': '/content/MCE/MCE_Dataset/Audio/1_MCE/1_3.wav', 'text': '"飛機已經被grounded喇，因為typhoon signal已經升到No.8嚟啦。"', 'topic': '天氣', 'duration': 6.81}


## 3. 转成 Hugging Face Dataset

读取配对结果，转成 16 kHz Audio 栏位，并切分 train / test（约 9:1）。


In [5]:
from datasets import Dataset, Audio
import json

with open("/content/mce_paired.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"总资料笔数：{len(data)}")

ds = Dataset.from_list(data)
ds = ds.cast_column("audio_path", Audio(sampling_rate=16000))
ds = ds.rename_column("audio_path", "audio")
ds = ds.rename_column("text", "sentence")

# 切分 train / test
ds = ds.train_test_split(test_size=0.1, seed=42)

print(ds)

# 储存（之后可直接 load_from_disk，不必重配对）
ds.save_to_disk("/content/mce_hf_dataset")
print("已储存到 /content/mce_hf_dataset")


总资料笔数：14048
DatasetDict({
    train: Dataset({
        features: ['id', 'audio', 'sentence', 'topic', 'duration'],
        num_rows: 12643
    })
    test: Dataset({
        features: ['id', 'audio', 'sentence', 'topic', 'duration'],
        num_rows: 1405
    })
})


Saving the dataset (0/8 shards):   0%|          | 0/12643 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1405 [00:00<?, ? examples/s]

已储存到 /content/mce_hf_dataset


## 4. Baseline 评估（原始 Whisper-small）

先用原始 `openai/whisper-small` 在 test set 上推理，计算 CER / WER 作为对照基准。


In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_from_disk
import numpy as np
import re
import evaluate

dataset = load_from_disk("/content/mce_hf_dataset")
test_dataset = dataset["test"]
print(f"Test set 笔数：{len(test_dataset)}")

model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to("cuda")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

def get_audio_array(audio_item):
    if hasattr(audio_item, "get_all_samples"):
        samples = audio_item.get_all_samples()
        array = samples.data.numpy().squeeze()
        sr = samples.sample_rate
    elif isinstance(audio_item, dict) and "array" in audio_item:
        array = audio_item["array"]
        sr = audio_item["sampling_rate"]
    else:
        array = np.array(audio_item)
        sr = 16000
    return array, sr

def clean_text(text):
    """清理文字: 去掉前后引号、多余空白"""
    if text is None:
        return ""
    text = str(text).strip()
    text = re.sub(r'^[\"\'「」『』]+|[\"\'「」『』]+$', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def map_to_pred(batch):
    array, sr = get_audio_array(batch["audio"])
    if array.ndim > 1:
        array = array.mean(axis=0)
    array = array.astype(np.float32)

    input_features = processor(
        array,
        sampling_rate=sr,
        return_tensors="pt"
    ).input_features.to("cuda")

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            language="zh",
            task="transcribe",
            max_length=225
        )

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    batch["prediction"] = transcription
    return batch

# 先预览 5 笔
print("先预览 5 笔结果...")
small_test = test_dataset.select(range(5))
result_small = small_test.map(map_to_pred, remove_columns=["audio"])
for i in range(5):
    print(f"【参考】{clean_text(result_small[i]['sentence'])}")
    print(f"【预测】{clean_text(result_small[i]['prediction'])}")
    print("-" * 60)

# 完整 Baseline 评估
print("\n开始完整 Baseline 评估（约需 15~25 分钟）...")
result = test_dataset.map(map_to_pred, remove_columns=["audio"])

predictions = [clean_text(x["prediction"]) for x in result]
references  = [clean_text(x["sentence"]) for x in result]

cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

cer = cer_metric.compute(predictions=predictions, references=references)
wer = wer_metric.compute(predictions=predictions, references=references)

print(f"\n========== Baseline 结果 ==========")
print(f"原始 Whisper-small 的 Character Error Rate (CER)：{cer * 100:.2f}%")
print(f"原始 Whisper-small 的 Word Error Rate (WER/MER)：{wer * 100:.2f}%")
print(f"测试笔数：{len(predictions)}")

ref_lens = [len(r) for r in references]
pred_lens = [len(p) for p in predictions]
print(f"参考文字平均长度：{sum(ref_lens)/len(ref_lens):.1f} 字")
print(f"预测文字平均长度：{sum(pred_lens)/len(pred_lens):.1f} 字")


Test set 笔数：1405


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

... (9 outputs left out)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


【参考】我有個好特別嘅愛好就係集郵，雖然聽落去有啲老土，但當你開始揀返啲stamp同了解背後嘅story，你就會覺得有好大嘅滿足感。
【预测】我有个很特别的爱好就是习游虽然听上去有些老土但当你开始一些Stamps和了解背后的故事你就会觉得很大满足感
------------------------------------------------------------
【参考】Eat more fruit and vegetables: 佢哋豐富嘅nutrients同流食纖維, 可以增強我哋嘅immune system同防止疾病.
【预测】Eat my fruit and vegetables.可以再次加上芥末和牛肉的味道,可以增加我们的美食和饱止疾病。
------------------------------------------------------------
【参考】漢堡和薯條係我最好嘅grab-and-go food！
【预测】漢武羅薯條是我最好的Grab & Go Foot
------------------------------------------------------------
【参考】Stay focused, 保持专注，才能够把工作做得更加完成。
【预测】Stay focused保持專注才能夠把工作做得更加完成
------------------------------------------------------------
【参考】學習可以係fun嘅，唔係悶嘅。將每樣事情變成game，喺其中find joy吧！
【预测】學習可以分的,不是悶的將每件事情變成遊戲,在其中再吧
------------------------------------------------------------

开始完整 Baseline 评估（约需 15~25 分钟）...


Map:   0%|          | 0/1405 [00:00<?, ? examples/s]

## 5. LoRA Fine-tuning

使用 PEFT LoRA（r=16，target: q_proj / v_proj）fine-tune Whisper-small。
训练设定与之前实验一致，方便比较。


In [ ]:
import torch
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model
from datasets import load_from_disk
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import numpy as np

# ====================== 1. 载入资料 ======================
dataset = load_from_disk("/content/mce_hf_dataset")
print(dataset)

# ====================== 2. 载入模型与 Processor ======================
model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name, language="zh", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_name)

# 设定生成参数
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.language = "zh"
model.generation_config.task = "transcribe"

# ====================== 3. 设定 LoRA ======================
lora_config = LoraConfig(
    r=16,                      # rank，T4/L4 建议 16
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # Whisper 常用这两个
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ====================== 4. 资料预处理 ======================
def prepare_dataset(batch):
    input_features = []
    labels = []

    for audio, sentence in zip(batch["audio"], batch["sentence"]):
        if hasattr(audio, "get_all_samples"):
            samples = audio.get_all_samples()
            array = samples.data.numpy().squeeze()
            sr = samples.sample_rate
        else:
            array = audio["array"]
            sr = audio["sampling_rate"]

        if array.ndim > 1:
            array = array.mean(axis=0)
        array = array.astype(np.float32)

        feats = processor(array, sampling_rate=sr).input_features[0]
        input_features.append(feats)
        labels.append(processor.tokenizer(sentence).input_ids)

    batch["input_features"] = input_features
    batch["labels"] = labels
    return batch

print("开始处理训练资料（这步会花一些时间）...")

# 清掉可能残留的损坏缓存
n_cleaned = dataset["train"].cleanup_cache_files()
print(f"已清理 {n_cleaned} 个快取档案")

train_dataset = dataset["train"].map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    batched=True,
    batch_size=8,
    desc="Processing train"
)

print("处理完成！")
print(train_dataset)

# ====================== 5. Data Collator ======================
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 输入特征
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 标签
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # 把 padding 的地方设成 -100（不算 loss）
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # 如果开头有 bos token，就拿掉（Whisper 特殊处理）
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# ====================== 6. 训练参数 ======================
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-mce-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,         # 有效 batch size = 16
    learning_rate=1e-4,
    warmup_steps=100,
    max_steps=1500,
    fp16=True,
    eval_strategy="no",
    save_steps=500,
    logging_steps=50,
    report_to=["none"],
    predict_with_generate=True,
    generation_max_length=225,
    save_total_limit=2,
    remove_unused_columns=False,
    label_names=["labels"],
)

# ====================== 7. Trainer ======================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
)

# ====================== 8. 开始训练 ======================
print("开始 LoRA Fine-tuning...")
trainer.train()

# 储存 LoRA 权重
trainer.save_model("./whisper-small-mce-lora-final")
processor.save_pretrained("./whisper-small-mce-lora-final")
print("训练完成！模型已储存到 ./whisper-small-mce-lora-final")


## 6. Fine-tuned 模型评估

载入 LoRA 权重、合并后在 test set 上计算 CER / WER，并与 Baseline 比较。


In [ ]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from datasets import load_from_disk
import evaluate
import numpy as np
import re

# 1. 载入资料
dataset = load_from_disk("/content/mce_hf_dataset")
test_dataset = dataset["test"]
print(f"Test set 笔数：{len(test_dataset)}")

# 2. 载入 Fine-tuned 模型
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model = PeftModel.from_pretrained(base_model, "./whisper-small-mce-lora-final")
model = model.merge_and_unload()  # 合并 LoRA 权重，方便推理
model = model.to("cuda")
model.eval()

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="zh", task="transcribe")

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# 3. 音讯读取与文字清理
def get_audio_array(audio_item):
    if hasattr(audio_item, "get_all_samples"):
        samples = audio_item.get_all_samples()
        array = samples.data.numpy().squeeze()
        sr = samples.sample_rate
    elif isinstance(audio_item, dict) and "array" in audio_item:
        array = audio_item["array"]
        sr = audio_item["sampling_rate"]
    else:
        array = np.array(audio_item)
        sr = 16000
    return array, sr

def clean_text(text):
    text = str(text).strip()
    text = re.sub(r'^[\"\'「」『』]+|[\"\'「」『』]+$', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 4. 预测函数
def map_to_pred(batch):
    array, sr = get_audio_array(batch["audio"])
    if array.ndim > 1:
        array = array.mean(axis=0)
    array = array.astype(np.float32)

    input_features = processor(
        array,
        sampling_rate=sr,
        return_tensors="pt"
    ).input_features.to("cuda")

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features,
            language="zh",
            task="transcribe",
            max_length=225
        )

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    batch["prediction"] = transcription
    return batch

# 5. 先看 5 笔效果
print("先预览 5 笔结果...")
small_test = test_dataset.select(range(5))
result_small = small_test.map(map_to_pred, remove_columns=["audio"])

for i in range(5):
    print(f"【参考】{clean_text(result_small[i]['sentence'])}")
    print(f"【预测】{clean_text(result_small[i]['prediction'])}")
    print("-" * 60)

# 6. 完整评估
print("\n开始完整评估 Fine-tuned 模型（约需 15~20 分钟）...")
result = test_dataset.map(map_to_pred, remove_columns=["audio"])

predictions = [clean_text(x["prediction"]) for x in result]
references  = [clean_text(x["sentence"]) for x in result]

cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

cer = cer_metric.compute(predictions=predictions, references=references)
wer = wer_metric.compute(predictions=predictions, references=references)

print(f"\n========== Fine-tuned 结果 ==========")
print(f"Fine-tuned Whisper-small (LoRA) CER：{cer * 100:.2f}%")
print(f"Fine-tuned Whisper-small (LoRA) WER：{wer * 100:.2f}%")

print(f"\n========== 与 Baseline 比较 ==========")
print(f"Baseline CER：        47.41%")
print(f"Fine-tuned CER：      {cer * 100:.2f}%")
print(f"绝对下降：            {47.41 - cer * 100:.2f} 百分点")
print(f"相对错误率下降：      {(47.41 - cer * 100) / 47.41 * 100:.1f}%")


## 7. 储存到 Google Drive

把模型、实验结果、数据集备份到 Drive，方便之后断线继续用。


In [ ]:
import os
import shutil
import json

save_dir = "/content/drive/MyDrive/COMP4330_Whisper_MCE"
os.makedirs(save_dir, exist_ok=True)

# 1. 复制 Fine-tuned 模型
src_model = "./whisper-small-mce-lora-final"
dst_model = os.path.join(save_dir, "whisper-small-mce-lora-final")

if os.path.exists(src_model):
    if os.path.exists(dst_model):
        shutil.rmtree(dst_model)
    shutil.copytree(src_model, dst_model)
    print(f"模型已储存到：{dst_model}")
else:
    print("找不到模型资料夹，请确认路径")

# 2. 储存实验结果
results = {
    "baseline_cer": 47.41,
    "finetuned_cer": 17.30,
    "absolute_improvement": 30.11,
    "relative_improvement": 63.5,
    "baseline_wer": 113.05,
    "finetuned_wer": 79.53,
    "test_samples": 1405,
    "train_samples": 12643,
    "model": "whisper-small + LoRA (r=16)",
    "dataset": "MCE (official)",
    "max_steps": 1500
}

results_path = os.path.join(save_dir, "experiment_results.json")
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"结果已储存到：{results_path}")
print(results)

# 3. 复制数据集（若尚未存在）
dataset_dst = os.path.join(save_dir, "mce_hf_dataset")
if not os.path.exists(dataset_dst):
    shutil.copytree("/content/mce_hf_dataset", dataset_dst)
    print(f"数据集已储存到：{dataset_dst}")
else:
    print("数据集资料夹已存在，跳过复制")


## 使用说明

| 区块 | 内容 | 何时需要重跑 |
|------|------|-------------|
| 1 | 环境 + 下载 | 新 Colab 会话 |
| 2–3 | 配对 + Dataset | 数据集已在 Drive 可跳过，改用 `load_from_disk` |
| 4 | Baseline | 已有结果可跳过 |
| 5 | LoRA 训练 | 已有模型可跳过 |
| 6 | Fine-tuned 评估 | 需要新数字时 |
| 7 | 存 Drive | 每次训练后建议执行 |

**从 Drive 继续**: 见 `whisper_resume_and_report.ipynb`。

**下一步（可选）**: QLoRA 对比实验。
